Если захочется продолжить поиск промптов с определённого места, то нужно знать откуда продолжать.  
В этом ноутбуке я написал поиск лучшей версии промпта и его скор на тестовых данных.  


В langfuse нельзя через python SDK достать метрики запусков, пробую ходить по API  
https://langfuse.com/docs/api-and-data-platform/features/public-api  

этот код возвращает пустой список, хотя скоры есть и они привязаны к run
```
run = client.api.datasets.get_run(dataset_name, "run-0")
client.api.score_v_2.get(dataset_run_id=run.id)
```


In [ ]:
import asyncio
import os
from urllib.parse import quote

from langfuse import get_client

os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-9058c552-1e8e-41c2-bcea-8abae84494da"
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-1434620a-ab84-4592-99f6-e3dd7419dd64"
os.environ["LANGFUSE_BASE_URL"] = "http://localhost:3000"

In [ ]:
client = get_client()

In [ ]:
dataset_name = quote("RusLawOD/test", safe="")
dataset_name

# 'RusLawOD%2Ftest'

In [ ]:
run = client.api.datasets.get_run(dataset_name, "run-0")
client.api.score_v_2.get(dataset_run_id=run.id)

# GetScoresResponse(data=[], meta=MetaResponse(page=1, limit=50, total_items=0, total_pages=0))

In [ ]:
runs = client.api.datasets.get_runs(dataset_name)
scores = []

for run in runs.data:
    run = client.api.datasets.get_run(dataset_name, run.name)

    run_scores = []
    for item in run.dataset_run_items:
        trace = client.api.trace.get(item.trace_id)
        score = trace.scores[0].value
        run_scores.append(score)

    scores.append(
            {
                'run_name': run.name,
                "score": sum(run_scores) / len(run_scores),
                "prompt_version": trace.observations[0].prompt_version,
            }
        )

In [ ]:
scores

# [{'run_name': 'run-4', 'score': 0.9142857142857143, 'prompt_version': 6},
#  {'run_name': 'run-3', 'score': 0.8428571428571429, 'prompt_version': 5},
#  {'run_name': 'run-2', 'score': 0.8428571428571429, 'prompt_version': 4},
#  {'run_name': 'run-1', 'score': 0.9285714285714285, 'prompt_version': 3},
#  {'run_name': 'run-0', 'score': 0.9, 'prompt_version': 2}]

In [ ]:
async def get_run_scores(dataset_name: str, run_name: str):
    run = await client.async_api.datasets.get_run(dataset_name, run_name)

    tasks = []
    for item in run.dataset_run_items:
        tasks.append(client.async_api.trace.get(item.trace_id))
    
    run_scores = []
    traces = await asyncio.gather(*tasks)
    for trace in traces:
        score = trace.scores[0].value
        run_scores.append(score)

    return sum(run_scores) / len(run_scores), trace.observations[0].prompt_version

async def get_dataset_scores(dataset_name: str):
    runs = await client.async_api.datasets.get_runs(dataset_name)
    scores = []

    tasks = [get_run_scores(dataset_name, run.name) for run in runs.data]
    results = await asyncio.gather(*tasks)
    for run, (score, prompt_version) in zip(runs.data, results):
        scores.append(
                {
                    "run_name": run.name,
                    "score": score,
                    "prompt_version": prompt_version,
                }
            )
    return scores    

In [ ]:
scores = await get_dataset_scores(dataset_name)

In [ ]:
scores

# [{'run_name': 'run-4', 'score': 0.9142857142857143, 'prompt_version': 6},
#  {'run_name': 'run-3', 'score': 0.8428571428571429, 'prompt_version': 5},
#  {'run_name': 'run-2', 'score': 0.8428571428571429, 'prompt_version': 4},
#  {'run_name': 'run-1', 'score': 0.9285714285714285, 'prompt_version': 3},
#  {'run_name': 'run-0', 'score': 0.9, 'prompt_version': 2}]